In [1]:
# --- Predict using the 9 selected features (by index) ---
import joblib, os, pandas as pd, numpy as np

# 1. Map indices -> column names (based on your earlier column order)
selected_indices = [4,5,9,16,20,21,22,23,24]
all_cols = [
 'Age','Gender','EducationBackground','MaritalStatus','EmpDepartment','EmpJobRole',
 'BusinessTravelFrequency','DistanceFromHome','EmpEducationLevel','EmpEnvironmentSatisfaction',
 'EmpHourlyRate','EmpJobInvolvement','EmpJobLevel','EmpJobSatisfaction','NumCompaniesWorked',
 'OverTime','EmpLastSalaryHikePercent','EmpRelationshipSatisfaction','TotalWorkExperienceInYears',
 'TrainingTimesLastYear','EmpWorkLifeBalance','ExperienceYearsAtThisCompany',
 'ExperienceYearsInCurrentRole','YearsSinceLastPromotion','YearsWithCurrManager','Attrition','ExperienceGap'
]

# Build feature list from indices
feature_names = [ all_cols[i] for i in selected_indices ]
print("Selected features (count={}):\n".format(len(feature_names)), feature_names)

# 2. Load model and data
model_path = "models/final_random_forest_model.joblib"
if not os.path.exists(model_path):
    raise FileNotFoundError(f"Saved model not found: {model_path}")

model = joblib.load(model_path)
df = pd.read_csv("data/processed/employee_perf_clean.csv")

# 3. Validate features exist in dataframe
missing = [c for c in feature_names if c not in df.columns]
if missing:
    raise ValueError("The following required features are missing from the data: " + str(missing))

# 4. Select feature columns in exact order
X_pred = df[feature_names].copy()
print("\nSelected X_pred shape:", X_pred.shape)
print("Dtypes before conversion:\n", X_pred.dtypes)

# 5. Ensure all columns are numeric (best-effort)
#    - If a column is object, convert to category codes.
#    - This is risky if training used a different mapping; prefer saved encoders.
for col in X_pred.columns:
    if X_pred[col].dtype == 'object' or str(X_pred[col].dtype).startswith('category'):
        print(f"Column '{col}' is object/category; converting to category codes (best-effort).")
        X_pred[col] = X_pred[col].astype('category').cat.codes

# Convert any remaining non-numeric to numeric if possible
X_pred = X_pred.apply(pd.to_numeric, errors='coerce')

# 6. Check for NaNs produced by conversion and handle them
nan_cols = X_pred.columns[X_pred.isna().any()].tolist()
if nan_cols:
    print("Warning: NaNs detected in columns after conversion:", nan_cols)
    # fill NaNs with median of that column as a sensible fallback
    for c in nan_cols:
        med = X_pred[c].median()
        X_pred[c].fillna(med, inplace=True)
    print("Filled NaNs with column medians.")

print("\nDtypes after conversion:\n", X_pred.dtypes)
print("\nSample of input features to model:")
display(X_pred.head())

# 7. Final shape check vs model
expected = getattr(model, "n_features_in_", None)
print(f"\nModel expects {expected} features.")
if expected is not None and X_pred.shape[1] != expected:
    raise ValueError(f"Feature count mismatch: model expects {expected}, but X_pred has {X_pred.shape[1]}")

# 8. Predict
preds = model.predict(X_pred)
df_out = df.copy()
df_out["PredictedPerformance"] = preds

# 9. Save predictions
os.makedirs("data/processed", exist_ok=True)
out_path = "data/processed/predicted_results.csv"
df_out.to_csv(out_path, index=False)
print(f"\nPredictions saved to: {out_path}")

# 10. Quick summary: distribution of predicted classes
print("\nPredicted class distribution:")
print(pd.Series(preds).value_counts(normalize=True))

# 11. Feature importances from the RandomForest (map to the selected features)
try:
    importances = model.feature_importances_
    fi = pd.Series(importances, index=feature_names).sort_values(ascending=False)
    print("\nTop 5 feature importances (for this model):")
    print(fi.head(5))
    # Save importances
    fi.to_csv("data/processed/feature_importances_used_for_prediction.csv", header=['importance'])
except Exception as e:
    print("Could not extract feature_importances_ from model:", e)


Selected features (count=9):
 ['EmpDepartment', 'EmpJobRole', 'EmpEnvironmentSatisfaction', 'EmpLastSalaryHikePercent', 'EmpWorkLifeBalance', 'ExperienceYearsAtThisCompany', 'ExperienceYearsInCurrentRole', 'YearsSinceLastPromotion', 'YearsWithCurrManager']

Selected X_pred shape: (1200, 9)
Dtypes before conversion:
 EmpDepartment                   int64
EmpJobRole                      int64
EmpEnvironmentSatisfaction      int64
EmpLastSalaryHikePercent        int64
EmpWorkLifeBalance              int64
ExperienceYearsAtThisCompany    int64
ExperienceYearsInCurrentRole    int64
YearsSinceLastPromotion         int64
YearsWithCurrManager            int64
dtype: object

Dtypes after conversion:
 EmpDepartment                   int64
EmpJobRole                      int64
EmpEnvironmentSatisfaction      int64
EmpLastSalaryHikePercent        int64
EmpWorkLifeBalance              int64
ExperienceYearsAtThisCompany    int64
ExperienceYearsInCurrentRole    int64
YearsSinceLastPromotion         i

,EmpDepartment,EmpJobRole,EmpEnvironmentSatisfaction,EmpLastSalaryHikePercent,EmpWorkLifeBalance,ExperienceYearsAtThisCompany,ExperienceYearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,5,13,4,12,2,10,7,0,8
1,5,13,4,12,3,7,7,1,7
2,5,13,4,21,3,18,13,1,12
3,3,8,2,15,2,21,6,12,6
4,5,13,1,14,3,2,2,2,2



Model expects 9 features.

Predictions saved to: data/processed/predicted_results.csv

Predicted class distribution:
4    0.945
3    0.055
Name: proportion, dtype: float64

Top 5 feature importances (for this model):
EmpLastSalaryHikePercent        0.292362
EmpEnvironmentSatisfaction      0.249270
YearsSinceLastPromotion         0.133319
EmpDepartment                   0.071165
ExperienceYearsInCurrentRole    0.062394
dtype: float64


C:\Users\sanjay\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(
